# Session 3: LangGraph Orchestration & Workflow Design (Student Notebook)

Welcome to Session 3! In this notebook, you will learn LangGraph -- a framework for building stateful, multi-step agentic workflows as directed graphs. You will model complex agent behaviors using nodes, edges, and shared state, enabling conditional routing, iterative refinement, and human-in-the-loop patterns.

**McKinsey Context:** Every demo and task in this session uses realistic consulting scenarios -- from engagement scoping pipelines and PE due diligence routing to iterative recommendation refinement and market entry assessments. You will see how LangGraph patterns map directly to how McKinsey structures its workflows.

## Learning Objectives

By the end of this session, you will be able to:

1. **Create StateGraphs** with typed state schemas to model consulting engagement pipelines
2. **Define nodes and edges** to represent workflow steps such as data gathering, market analysis, and recommendation formulation
3. **Add conditional edges** for dynamic routing based on consulting decisions (engagement complexity, client tier, industry sector)
4. **Build cyclic workflows** for iterative refinement until deliverables reach partner-quality standards
5. **Implement a ReAct agent** as a McKinsey analyst researching markets, analyzing financials, and benchmarking competitors
6. **Add human-in-the-loop** checkpoints for partner review and client sign-off

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
import operator
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on. Each demo uses a McKinsey consulting scenario to illustrate the LangGraph concept.

### Demo 1: LangGraph Basics -- Modeling a Consulting Engagement Pipeline

A StateGraph has three core components:
- **State**: A TypedDict that holds all data flowing through the graph (think of it as the engagement tracker)
- **Nodes**: Python functions that read and update state (each node is a step in the consulting process)
- **Edges**: Connections that define the execution order (the workflow sequence)

In this demo, we model a simple engagement intake pipeline: a client request arrives, we extract the scope, classify the industry sector, and format a preliminary engagement brief.

In [ ]:
# Demo 1 - LangGraph Basics: Consulting Engagement Pipeline

# Step 1: Define the state schema (the engagement tracker)
class EngagementState(TypedDict):
    client_request: str
    scope_summary: str
    industry_sector: str
    engagement_brief: str

# Step 2: Define node functions (each takes state, returns partial state update)
def extract_scope_node(state: EngagementState) -> dict:
    """Extract and normalize the engagement scope from the client request."""
    return {"scope_summary": state["client_request"].strip().upper()}

def classify_industry_node(state: EngagementState) -> dict:
    """Classify the industry sector based on keywords in the request."""
    text = state["client_request"].lower()
    if any(kw in text for kw in ["bank", "financial", "fintech", "insurance"]):
        sector = "Financial Services"
    elif any(kw in text for kw in ["pharma", "health", "biotech", "hospital"]):
        sector = "Healthcare & Life Sciences"
    elif any(kw in text for kw in ["retail", "consumer", "e-commerce", "cpg"]):
        sector = "Consumer & Retail"
    else:
        sector = "Cross-Industry"
    return {"industry_sector": sector}

def format_brief_node(state: EngagementState) -> dict:
    """Format the preliminary engagement brief."""
    return {"engagement_brief": f"ENGAGEMENT BRIEF\nScope: {state['scope_summary']}\nSector: {state['industry_sector']}"}

# Step 3: Build the graph
graph = StateGraph(EngagementState)

# Add nodes (each node represents a consulting process step)
graph.add_node("extract_scope", extract_scope_node)
graph.add_node("classify_industry", classify_industry_node)
graph.add_node("format_brief", format_brief_node)

# Add edges (linear flow: scope -> classify -> brief)
graph.set_entry_point("extract_scope")
graph.add_edge("extract_scope", "classify_industry")
graph.add_edge("classify_industry", "format_brief")
graph.add_edge("format_brief", END)

# Step 4: Compile and run
app = graph.compile()

result = app.invoke({
    "client_request": "Our fintech startup needs help with market entry strategy for Southeast Asia",
    "scope_summary": "", "industry_sector": "", "engagement_brief": ""
})
print("Engagement Pipeline Result:")
print(f"  Client Request  : {result['client_request']}")
print(f"  Scope Summary   : {result['scope_summary']}")
print(f"  Industry Sector : {result['industry_sector']}")
print(f"  Brief           : {result['engagement_brief']}")

### Demo 2: Conditional Edges -- Routing Client Engagements by Complexity

Conditional edges let the graph take different paths based on the current state. This is how consulting workflows adapt -- a quick benchmarking study follows a different path than a full-scale transformation program.

Here we use an LLM to classify engagement complexity and route to the appropriate workstream: **rapid assessment**, **standard engagement**, or **transformation program**.

In [ ]:
# Demo 2 - Conditional Edges: Routing Engagements by Complexity

class EngagementRouterState(TypedDict):
    engagement_request: str
    complexity: str
    workstream_output: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def assess_complexity_node(state: EngagementRouterState) -> dict:
    """Use LLM to classify engagement complexity."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager. Classify this client request into exactly one category: 'rapid' (2-4 week benchmarking or quick scan), 'standard' (8-12 week strategy or operations engagement), or 'transformation' (6+ month large-scale change program). Respond with just the one word."),
        HumanMessage(content=state["engagement_request"])
    ])
    complexity = response.content.strip().lower()
    print(f"  Complexity assessed: {complexity}")
    return {"complexity": complexity}

def rapid_assessment_node(state: EngagementRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey associate conducting a rapid assessment. Provide a concise 2-week diagnostic framework with key hypotheses to test."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[RAPID ASSESSMENT]\n{response.content}"}

def standard_engagement_node(state: EngagementRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager designing a standard 10-week engagement. Outline the workplan with key phases: diagnostic, analysis, recommendations, and implementation roadmap."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[STANDARD ENGAGEMENT]\n{response.content}"}

def transformation_node(state: EngagementRouterState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner designing a large-scale transformation program. Outline the multi-phase approach including change management, capability building, and performance tracking."),
        HumanMessage(content=state["engagement_request"])
    ])
    return {"workstream_output": f"[TRANSFORMATION PROGRAM]\n{response.content}"}

# Routing function -- decides which workstream to activate
def route_by_complexity(state: EngagementRouterState) -> str:
    c = state["complexity"]
    if "rapid" in c:
        return "rapid_assessment"
    elif "transformation" in c:
        return "transformation"
    else:
        return "standard_engagement"

# Build graph
graph = StateGraph(EngagementRouterState)
graph.add_node("assess_complexity", assess_complexity_node)
graph.add_node("rapid_assessment", rapid_assessment_node)
graph.add_node("standard_engagement", standard_engagement_node)
graph.add_node("transformation", transformation_node)

graph.set_entry_point("assess_complexity")
graph.add_conditional_edges("assess_complexity", route_by_complexity, {
    "rapid_assessment": "rapid_assessment",
    "standard_engagement": "standard_engagement",
    "transformation": "transformation"
})
graph.add_edge("rapid_assessment", END)
graph.add_edge("standard_engagement", END)
graph.add_edge("transformation", END)

app = graph.compile()

# Test with different engagement requests
requests = [
    "We need a quick competitive benchmarking of our pricing vs. top 3 rivals",
    "Help us design a new go-to-market strategy for entering the European healthcare market",
    "We are a $20B conglomerate and need to completely restructure our operating model across 15 business units"
]

for req in requests:
    print(f"\nRequest: {req}")
    result = app.invoke({"engagement_request": req, "complexity": "", "workstream_output": ""})
    print(f"Output: {result['workstream_output'][:200]}...\n{'='*60}")

### Demo 3: Building a ReAct Agent -- McKinsey Market Research Analyst

The ReAct (Reason + Act) pattern models how a McKinsey analyst works:
1. **Think** -- Reason about what data is needed (market size? competitive landscape? margins?)
2. **Act** -- Search for market intelligence using available tools
3. **Observe** -- Process the research findings
4. **Repeat** until a well-supported answer emerges

Our analyst has access to simulated tools for market data, financial metrics, and competitive intelligence.

In [ ]:
# Demo 3 - ReAct Agent: McKinsey Market Research Analyst

class ReActState(TypedDict):
    question: str
    thoughts: list[str]
    actions: list[str]
    observations: list[str]
    final_answer: str
    step_count: int

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# Simulated McKinsey research tools
def research_tool(query):
    """Simulated market intelligence database."""
    data = {
        "ev market size": "The global EV market was valued at $388B in 2024, projected to reach $950B by 2030 (CAGR ~16%). China leads with 60% market share.",
        "tesla competitors": "Key Tesla competitors: BYD (largest global EV seller by volume), Volkswagen Group (ID. series), Hyundai-Kia (fastest growing in Europe), and Rivian (US truck segment).",
        "ev margins": "Average EV gross margins: Tesla ~18%, BYD ~20%, legacy OEMs ~5-8% on EVs. Battery costs represent 30-40% of vehicle cost.",
        "southeast asia market": "Southeast Asia EV penetration is ~5% (2024), led by Thailand (govt subsidies) and Indonesia (nickel supply chain). Market expected to grow 25% CAGR through 2030.",
        "mckinsey frameworks": "Key McKinsey frameworks: MECE (Mutually Exclusive, Collectively Exhaustive), 7S Model, Three Horizons, Profit Pools, and Value Chain Analysis."
    }
    for key, val in data.items():
        if key in query.lower():
            return val
    return f"No data found for: {query}"

def think_node(state: ReActState) -> dict:
    """McKinsey analyst reasons about what data to gather next."""
    context = ""
    for i in range(len(state["actions"])):
        context += f"Action: {state['actions'][i]}\nObservation: {state['observations'][i]}\n"
    
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey research analyst. Given the question and any previous research findings, decide your next step. Either search for more data (respond: ACTION: search '<query>') or provide a final synthesized answer (respond: FINAL ANSWER: <answer>). Think like a consultant -- be MECE in your approach."),
        HumanMessage(content=f"Research question: {state['question']}\n\n{context}\nWhat should I research next?")
    ])
    thought = response.content
    print(f"  Analyst thinks: {thought[:120]}...")
    return {"thoughts": state["thoughts"] + [thought]}

def act_node(state: ReActState) -> dict:
    """Execute the research action."""
    latest_thought = state["thoughts"][-1]
    
    if "FINAL ANSWER:" in latest_thought:
        answer = latest_thought.split("FINAL ANSWER:")[-1].strip()
        return {"final_answer": answer, "step_count": state["step_count"] + 1}
    
    if "ACTION: search" in latest_thought:
        query = latest_thought.split("search")[-1].strip().strip("'\"")
        observation = research_tool(query)
        print(f"  Research: search('{query}') -> {observation[:80]}...")
        return {
            "actions": state["actions"] + [f"search('{query}')"],
            "observations": state["observations"] + [observation],
            "step_count": state["step_count"] + 1
        }
    
    return {"final_answer": latest_thought, "step_count": state["step_count"] + 1}

def should_continue(state: ReActState) -> str:
    if state["final_answer"] or state["step_count"] >= 5:
        return "end"
    return "think"

# Build graph
graph = StateGraph(ReActState)
graph.add_node("think", think_node)
graph.add_node("act", act_node)

graph.set_entry_point("think")
graph.add_edge("think", "act")
graph.add_conditional_edges("act", should_continue, {"think": "think", "end": END})

app = graph.compile()

# Test: Ask the analyst a market research question
result = app.invoke({
    "question": "What is the EV market size and who are Tesla's main competitors?",
    "thoughts": [], "actions": [], "observations": [],
    "final_answer": "", "step_count": 0
})

print(f"\nAnalyst's Answer: {result['final_answer']}")
print(f"Research steps taken: {result['step_count']}")

### Demo 4: Cycles for Iterative Refinement -- Refining a Recommendation Until Partner-Quality

Cycles allow a graph to loop back and improve its output. In McKinsey, deliverables go through multiple rounds of refinement before they reach partner-quality. This demo models that iterative process: draft a strategic recommendation, have a simulated "partner review" critique it, and loop back to refine until it meets the bar or reaches max iterations.

In [ ]:
# Demo 4 - Cycles: Refining a Recommendation Until Partner-Quality

class RecommendationState(TypedDict):
    strategic_question: str
    draft_recommendation: str
    partner_feedback: str
    iteration: int
    is_partner_quality: bool

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

def formulate_recommendation(state: RecommendationState) -> dict:
    """Draft or refine a strategic recommendation."""
    if state["draft_recommendation"]:
        prompt = f"You are a McKinsey engagement manager. Refine this strategic recommendation based on partner feedback. Make it more MECE, insight-driven, and actionable.\n\nCurrent recommendation: {state['draft_recommendation']}\n\nPartner feedback: {state['partner_feedback']}\n\nProvide an improved recommendation (3-4 sentences, structured and concise):"
    else:
        prompt = f"You are a McKinsey engagement manager. Draft a strategic recommendation for this question (3-4 sentences). Be specific, actionable, and data-driven.\n\nStrategic question: {state['strategic_question']}"
    
    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [Iteration {state['iteration'] + 1}] Recommendation: {response.content[:100]}...")
    return {"draft_recommendation": response.content, "iteration": state["iteration"] + 1}

def partner_review(state: RecommendationState) -> dict:
    """Simulated partner review -- critiques the recommendation for quality."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner reviewing a strategic recommendation. Rate it 1-10 on: (1) MECE structure, (2) actionability, (3) insight depth. If average is 8+, respond with 'APPROVED - ready for client'. Otherwise, provide specific feedback on what to improve."),
        HumanMessage(content=state["draft_recommendation"])
    ])
    feedback = response.content
    is_approved = "APPROVED" in feedback.upper() or state["iteration"] >= 3
    print(f"  [Partner Review] {'PARTNER APPROVED' if is_approved else feedback[:80]}...")
    return {"partner_feedback": feedback, "is_partner_quality": is_approved}

def should_refine(state: RecommendationState) -> str:
    if state["is_partner_quality"]:
        return "end"
    return "formulate_recommendation"

# Build graph with cycle
graph = StateGraph(RecommendationState)
graph.add_node("formulate_recommendation", formulate_recommendation)
graph.add_node("partner_review", partner_review)

graph.set_entry_point("formulate_recommendation")
graph.add_edge("formulate_recommendation", "partner_review")
graph.add_conditional_edges("partner_review", should_refine, {
    "formulate_recommendation": "formulate_recommendation", "end": END
})

app = graph.compile()

result = app.invoke({
    "strategic_question": "Should our client, a mid-size European bank, acquire a fintech payments startup to accelerate digital transformation?",
    "draft_recommendation": "", "partner_feedback": "", "iteration": 0, "is_partner_quality": False
})

print(f"\nFinal Recommendation (after {result['iteration']} iterations):")
print(result["draft_recommendation"])

### Demo 5: Human-in-the-Loop -- Partner Sign-Off Before Client Delivery

In consulting, no major deliverable goes to the client without partner approval. LangGraph supports this through interrupt points where the graph pauses and waits for human approval before continuing. This demo models that gate: the system generates a client-ready action plan, pauses for partner review (simulated), and only proceeds to "deliver to client" if approved.

In [ ]:
# Demo 5 - Human-in-the-Loop: Partner Sign-Off Before Client Delivery

# Note: Full checkpointing requires a persistence backend (e.g., SQLite).
# This demo simulates the pattern without external dependencies.

class ClientDeliveryState(TypedDict):
    engagement_context: str
    action_plan: str
    partner_approved: bool
    delivery_output: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def prepare_deliverable(state: ClientDeliveryState) -> dict:
    """Generate a client-ready action plan."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey engagement manager. Create a structured 3-step action plan for the client. Include specific timelines, owners, and expected impact for each step."),
        HumanMessage(content=state["engagement_context"])
    ])
    print(f"  Deliverable prepared: {response.content[:120]}...")
    return {"action_plan": response.content}

def partner_signoff_node(state: ClientDeliveryState) -> dict:
    """Simulate partner review gate (auto-approve for demo)."""
    print(f"  [PARTNER REVIEW] Reviewing action plan for client delivery...")
    print(f"    Plan preview: {state['action_plan'][:200]}...")
    # In production, this would pause and wait for actual partner input
    approved = True  # Simulated approval
    print(f"  [PARTNER] {'Approved for client delivery' if approved else 'Needs revision -- send back to team'}")
    return {"partner_approved": approved}

def deliver_to_client(state: ClientDeliveryState) -> dict:
    """Package and deliver the final output to the client."""
    if not state["partner_approved"]:
        return {"delivery_output": "Delivery blocked -- partner approval required."}
    
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey partner delivering recommendations to a C-suite client. Wrap this action plan in a professional executive summary with a compelling opening and clear next steps."),
        HumanMessage(content=f"Action Plan:\n{state['action_plan']}")
    ])
    return {"delivery_output": response.content}

def route_after_review(state: ClientDeliveryState) -> str:
    return "deliver_to_client" if state["partner_approved"] else "end"

# Build graph
graph = StateGraph(ClientDeliveryState)
graph.add_node("prepare_deliverable", prepare_deliverable)
graph.add_node("partner_signoff", partner_signoff_node)
graph.add_node("deliver_to_client", deliver_to_client)

graph.set_entry_point("prepare_deliverable")
graph.add_edge("prepare_deliverable", "partner_signoff")
graph.add_conditional_edges("partner_signoff", route_after_review, {
    "deliver_to_client": "deliver_to_client", "end": END
})
graph.add_edge("deliver_to_client", END)

app = graph.compile()

result = app.invoke({
    "engagement_context": "A Fortune 500 CPG company needs to reduce supply chain costs by 15% within 18 months while maintaining service levels",
    "action_plan": "", "partner_approved": False, "delivery_output": ""
})

print(f"\nClient Delivery:\n{result['delivery_output'][:300]}...")

---
## Tasks (Your Turn!)

Now it is your turn to practice. Each task below has a description, hints, and a code skeleton with `TODO` placeholders. Fill in the code to complete each task. All tasks use McKinsey consulting scenarios.

### Task 1: Build a Client Onboarding Pipeline (Linear Workflow)

**Scenario:** When a new client engagement kicks off, McKinsey runs a standard onboarding pipeline: (1) gather and clean the raw client data, (2) generate a preliminary diagnostic summary using an LLM, (3) translate the diagnostic into an executive-ready brief.

Create a three-node linear graph that processes raw client data through: **gather_data** (clean/normalize), **analyze_situation** (LLM diagnostic), and **prepare_brief** (LLM executive brief).

In [ ]:
# Task 1 - Build a Simple Linear Workflow with LangGraph

# TODO: Create a StateGraph with three nodes:
# 1. clean_node: Remove extra whitespace and normalize the text
# 2. summarize_node: Use the LLM to summarize the cleaned text
# 3. translate_node: Use the LLM to translate the summary to Spanish
#
# Hint: Define a TypedDict state with fields: raw_text, clean_text, summary, translation
# Hint: Use graph.set_entry_point() and graph.add_edge() for linear flow
# Hint: Each node returns a dict with only the fields it updates

# YOUR CODE HERE


# Test your workflow
# result = app.invoke({...})
# print(f"Summary: {result['summary']}")
# print(f"Translation: {result['translation']}")

### Task 2: Create a Conditional Routing Agent

Build a support agent that classifies incoming messages (billing, technical, general) and routes them to specialized handler nodes.

In [ ]:
# Task 2 - Create a Conditional Routing Agent

# TODO: Build a support agent with:
# 1. A classify_node that uses the LLM to categorize the message
# 2. Three handler nodes: billing_node, technical_node, general_node
# 3. Conditional edges that route based on the classification
#
# Hint: Use add_conditional_edges() with a routing function
# Hint: Each handler should use a different system prompt persona
# Hint: All handlers should connect to END

# YOUR CODE HERE


# Test your agent
# result = app.invoke({"message": "I was charged twice for my subscription", ...})
# print(result["response"])

### Task 3: Implement a Self-Correcting Code Generation Workflow

Build a workflow that generates Python code, tests it (simulated), and if it fails, loops back to fix the code — up to 3 attempts.

In [ ]:
# Task 3 - Implement a Self-Correcting Code Generation Workflow

# TODO: Build a cyclic graph with:
# 1. generate_node: LLM generates Python code for the given task
# 2. test_node: Simulated testing (use exec() or simple string checks)
# 3. fix_node: LLM receives the error and fixes the code
# 4. Conditional edge: if tests pass -> END, else -> fix_node -> test_node
# 5. Max 3 iterations to prevent infinite loops
#
# Hint: State should include: task, code, test_result, passed, attempts
# Hint: Use a conditional edge after test_node to decide pass/retry
# Hint: The fix_node should receive the error message as context

# YOUR CODE HERE


# Test your workflow
# result = app.invoke({"task": "Write a function that checks if a number is prime", ...})
# print(f"Code:\n{result['code']}")
# print(f"Passed: {result['passed']} (attempts: {result['attempts']})")

### Task 4: Build a Research Agent with Planning and Execution Nodes

Build an agent that first plans a research strategy, then executes each step, and finally synthesizes the findings into a report.

In [ ]:
# Task 4 - Build a Research Agent with Planning and Execution Nodes

# TODO: Build a multi-stage research agent with:
# 1. plan_node: LLM generates a 3-step research plan (as JSON list)
# 2. execute_node: LLM executes the current step (simulated research)
# 3. check_node: Checks if all steps are complete
# 4. synthesize_node: LLM combines all findings into a final report
#
# Hint: State should include: topic, plan (list), current_step, findings (list), report
# Hint: execute_node processes one step at a time and increments current_step
# Hint: check_node routes back to execute if more steps remain, else to synthesize

# YOUR CODE HERE


# Test your agent
# result = app.invoke({"topic": "The impact of LLMs on software development", ...})
# print(f"Research Report:\n{result['report']}")

---
## Summary

In this session you learned how to orchestrate complex agentic workflows with LangGraph:

1. **StateGraph Basics** -- How to define typed state, add nodes as Python functions, and connect them with edges.
2. **Conditional Routing** -- How to use conditional edges so the graph takes different paths based on LLM decisions.
3. **ReAct Agents** -- How to implement the Reason-Act-Observe loop as a cyclic graph for tool-using agents.
4. **Iterative Refinement** -- How cycles enable self-correcting workflows that improve output quality over multiple passes.
5. **Human-in-the-Loop** -- How to add pause points for human oversight before critical actions.

**Up Next:** In Session 4, we will combine everything into multi-agent architectures — supervisor-worker patterns, agent handoffs, and collaborative problem-solving systems.